# RA Cohort Construction via MAP Phenotyping

End-to-end pipeline for building a Rheumatoid Arthritis (RA) cohort from MIMIC-IV
using Multimodal Automated Phenotyping (MAP).

**Sections:**
1. **Cohort Definition** — Query structured EHR data and build observation log
2. **NLP Features** — Extract CUI mentions from discharge notes via MedSpaCy
3. **MAP Phenotyping** — Run MAP algorithm for case/control classification
4. **Characterization** — Demographics, comorbidities, CONSORT flow, feature prevalence

**Anchor PheCode:** 714.1 (Rheumatoid Arthritis)

In [26]:
import os
import sys
import json
import warnings
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path(os.path.abspath("")).parent
MAPPING_DIR = PROJECT_ROOT / "mapping_dicts"
OUTPUT_DIR = PROJECT_ROOT / "output" / "ra-cohort-notebook"

DB_PATH = Path.home() / "m4" / "m4_data" / "databases" / "mimic_iv.duckdb"
NOTE_DB = Path.home() / "m4" / "m4_data" / "databases" / "mimic_iv_note.duckdb"

ONCE_CODIFIED = str(
    PROJECT_ROOT / "input" / "ONCE_Rheumatoid Arthritis_PheCode714.1_cos0.165.csv"
)
ONCE_NARRATIVE = str(
    PROJECT_ROOT
    / "input"
    / "ONCE_Rheumatoid Arthritis_C0003873_titlecos0.5_titlecut0.3_exactFALSE.csv"
)

MAIN_PHECODE = "714.1"

# Add source to path
for _p in ["preprocessing", "map"]:
    _full = str(PROJECT_ROOT / "src" / _p)
    if _full not in sys.path:
        sys.path.insert(0, _full)

# Create output directories
(OUTPUT_DIR / "data").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir:   {OUTPUT_DIR}")

Project root: /Users/irithkatiyar/Desktop/bmi702/bmi702-project
Output dir:   /Users/irithkatiyar/Desktop/bmi702/bmi702-project/output/ra-cohort-notebook


---
## Section 1: Cohort Definition

Query MIMIC-IV for diagnoses, prescriptions, and procedures. Roll up to standard
vocabularies (PheCodes, RxNorm, CCS) and build the structured observation log.

In [2]:
from preprocessing import build_obs_log

# ── Query raw tables from DuckDB ──────────────────────────────────────────────
conn = duckdb.connect(str(DB_PATH), read_only=True)

diagnoses = conn.execute("""
    SELECT d.subject_id, d.hadm_id, d.icd_code, d.icd_version, a.admittime
    FROM mimiciv_hosp.diagnoses_icd d
    INNER JOIN mimiciv_hosp.admissions a ON d.hadm_id = a.hadm_id
""").fetchdf()

prescriptions = conn.execute("""
    SELECT subject_id, hadm_id, drug, ndc, starttime, drug_type
    FROM mimiciv_hosp.prescriptions
    WHERE drug_type = 'MAIN'
""").fetchdf()

hcpcs = conn.execute("""
    SELECT subject_id, hadm_id, chartdate, hcpcs_cd
    FROM mimiciv_hosp.hcpcsevents
""").fetchdf()

patients = conn.execute("""
    SELECT subject_id, gender, anchor_age, anchor_year, anchor_year_group, dod
    FROM mimiciv_hosp.patients
""").fetchdf()

admissions = conn.execute("""
    SELECT subject_id, hadm_id, admittime, dischtime, race
    FROM mimiciv_hosp.admissions
""").fetchdf()

conn.close()

print(f"Diagnoses rows:     {len(diagnoses):,}")
print(f"Prescriptions rows: {len(prescriptions):,}")
print(f"HCPCS rows:         {len(hcpcs):,}")
print(f"Patients:           {len(patients):,}")
print(f"Admissions:         {len(admissions):,}")

Diagnoses rows:     6,364,488
Prescriptions rows: 16,791,812
HCPCS rows:         186,074
Patients:           364,627
Admissions:         546,028


In [3]:
# ── Build observation log (structured EHR only, no notes) ─────────────────────
obs_log = build_obs_log(
    icd_df=diagnoses,
    icd_col="icd_code",
    icd_date_col="admittime",
    drug_df=prescriptions,
    drug_ndc_col="ndc",
    drug_date_col="starttime",
    drug_col="drug",
    cpt_df=hcpcs,
    cpt_col="hcpcs_cd",
    cpt_date_col="chartdate",
    subject_col="subject_id",
    icd_mapping_file=str(MAPPING_DIR / "Phecode_map_v1_2_icd9_icd10cm.csv"),
    cpt_mapping_file=str(MAPPING_DIR / "CCS_Services_Procedures_v2025-1_052425.csv"),
    ndc_mapping_file=str(MAPPING_DIR / "ndc_to_rxnorm_ingredient.csv"),
    drug_name_mapping_file=str(MAPPING_DIR / "drug_name_to_rxnorm_ingredient.csv"),
)

print(
    f"\nObservation log: {len(obs_log):,} rows, {obs_log['subject_id'].nunique()} patients"
)
print("\nEvent type breakdown:")
print(obs_log["event_type"].value_counts().to_string())

# ── Report RA candidates (PheCode 714.1) ─────────────────────────────────────
ra_events = obs_log[obs_log["event"] == "PheCode:714.1"]
ra_subjects = ra_events["subject_id"].unique()
print(f"\nRA candidates (PheCode 714.1): {len(ra_subjects)} patients")
print(f"Total RA diagnosis events:     {len(ra_events)}")


Observation log: 21,700,111 rows, 223233 patients

Event type breakdown:
event_type
rxnorm     15649801
phecode     5943457
ccs          106853

RA candidates (PheCode 714.1): 3477 patients
Total RA diagnosis events:     8236


In [4]:
# ── Save structured data ──────────────────────────────────────────────────────
obs_log.to_parquet(OUTPUT_DIR / "data" / "obs_log.parquet", index=False)
patients.to_parquet(OUTPUT_DIR / "data" / "patients.parquet", index=False)
admissions.to_parquet(OUTPUT_DIR / "data" / "admissions.parquet", index=False)

print("Saved obs_log.parquet, patients.parquet, admissions.parquet to data/.")

Saved obs_log.parquet, patients.parquet, admissions.parquet to data/.


---
## Section 2: NLP Features

Extract CUI mentions from discharge notes for RA candidate patients using MedSpaCy.
Target CUIs come from the ONCE narrative file. Results are appended to the observation log.

In [5]:
from once import get_once_features
from preprocessing import notes_to_events

# ── Load ONCE features ────────────────────────────────────────────────────────
once_features = get_once_features(
    codified_file=ONCE_CODIFIED,
    narrative_file=ONCE_NARRATIVE,
)

target_cuis = once_features["nlp_target_cuis"]
print(f"Target CUIs for NLP: {len(target_cuis)}")
for tc in target_cuis[:5]:
    print(f"  {tc['cui']}: {tc['term']}")

# ── Identify candidates (patients with any ONCE codified event) ───────────────
once_events = set(once_features["codified_list"])
candidate_ids = sorted(
    obs_log[obs_log["event"].isin(once_events)]["subject_id"].unique()
)
print(f"\nCandidates with ONCE codified events: {len(candidate_ids)}")

Target CUIs for NLP: 442
  C0003873: Rheumatoid Arthritis
  C0035450: Rheumatoid Nodule
  C0409651: Seropositive rheumatoid arthritis
  C0574941: Inflamed joint
  C0242708: Antirheumatic Drugs, Disease-Modifying

Candidates with ONCE codified events: 12476


In [6]:
# ── Fetch discharge notes for candidates (batched) ───────────────────────────
conn = duckdb.connect(str(NOTE_DB), read_only=True)

BATCH_SIZE = 400
batches = [
    candidate_ids[i : i + BATCH_SIZE] for i in range(0, len(candidate_ids), BATCH_SIZE)
]

chunks = []
for i, batch in enumerate(batches):
    id_list = ", ".join(str(sid) for sid in batch)
    chunk = conn.execute(f"""
        SELECT subject_id, text, charttime
        FROM mimiciv_note.discharge
        WHERE subject_id IN ({id_list})
    """).fetchdf()
    chunks.append(chunk)
    if (i + 1) % 10 == 0:
        print(f"  Fetched batch {i + 1}/{len(batches)}")

conn.close()

notes_df = pd.concat(chunks, ignore_index=True).dropna(subset=["text"])
print(
    f"\nNotes fetched: {len(notes_df):,} ({notes_df['subject_id'].nunique()} patients)"
)

  Fetched batch 10/32
  Fetched batch 20/32
  Fetched batch 30/32

Notes fetched: 41,683 (10320 patients)


In [7]:
# ── Run MedSpaCy NER ──────────────────────────────────────────────────────────
# n_process=1 in Jupyter to avoid fork deadlocks
print("Running MedSpaCy NER (this may take several minutes)...")
nlp_obs = notes_to_events(
    notes_df=notes_df,
    text_col="text",
    date_col="charttime",
    target_cuis=target_cuis,
    subject_col="subject_id",
    max_note_chars=10_000,
    notes_per_patient=3,
    n_process=os.cpu_count() - 1,
)

print(
    f"\nNLP observations: {len(nlp_obs):,} rows, {nlp_obs['subject_id'].nunique()} patients"
)
print("\nCUI breakdown:")
print(nlp_obs["event"].value_counts().head(10).to_string())

Running MedSpaCy NER (this may take several minutes)...
  Parallel NLP: 9 workers × ~2411 notes


  Worker 0:   0%|          | 0/2411 [00:00<?, ?note/s]














  Worker 0:   0%|          | 1/2411 [00:00<15:00,  2.68note/s]

































  Worker 0:   0%|          | 2/2411 [00:00<15:11,  2.64note/s]



























  Worker 0:   0%|          | 3/2411 [00:01<16:26,  2.44note/s]































  Worker 0:   0%|          | 4/2411 [00:01<17:04,  2.35note/s]












  Worker 2:   0%|          | 5/2411 [00:01<13:51,  2.89note/s]

  Worker 3:   0%|          | 4/2411 [00:01<17:29,  2.29note/s]





  Worker 7:   0%|          | 4/2411 [00:01<15:48,  2.54note/s]







  Worker 0:   0%|          | 5/2411 [00:02<17:34,  2.28note/s]


















  Worker 8:   0%|          | 4/2408 [00:01<18:07,  2.21note/s]











  Worker 0:   0%|          | 6/2411 [00:02<18:10,  2.20note/s]



























  Worker 0:   0%|          | 7/2411 [00:02<16:41,  2.40note/s]






























































  Wo


NLP observations: 525,670 rows, 10320 patients

CUI breakdown:
event
CUI:C0030193    118000
CUI:C0012634     21381
CUI:C0013604     20894
CUI:C5201148     19211
CUI:C3714514     15643
CUI:C0000970     14852
CUI:C0032952     13699
CUI:C1261287     12395
CUI:C0003232     12022
CUI:C0038999     11877


In [14]:
# ── Combine structured + NLP observations and save ────────────────────────────
obs_log_with_nlp = pd.concat([obs_log, nlp_obs], ignore_index=True)
obs_log_with_nlp.to_parquet(
    OUTPUT_DIR / "data" / "obs_log_with_nlp.parquet", index=False
)

print(f"Combined obs log: {len(obs_log_with_nlp):,} rows")
print("Event type breakdown:")
print(obs_log_with_nlp["event_type"].value_counts().to_string())
print("\nSaved obs_log_with_nlp.parquet.")

Combined obs log: 22,225,781 rows
Event type breakdown:
event_type
rxnorm     15649801
phecode     5943457
cui          525670
ccs          106853

Saved obs_log_with_nlp.parquet.


---
## Section 3: MAP Phenotyping

Preprocess the observation log into a feature matrix, then run the MAP algorithm
(Poisson mixture model via R) to produce per-patient scores and binary case/control labels.

In [15]:
from map import preprocess_map, run_map

# ── Preprocess for MAP ────────────────────────────────────────────────────────
mat_df, note_df = preprocess_map(
    obs_log=obs_log_with_nlp,
    admissions_df=admissions,
    once_features=once_features,
    main_phecode=MAIN_PHECODE,
    subject_col="subject_id",
    min_nonzero=20,
)

print(f"MAP input matrix: {mat_df.shape[0]} patients x {mat_df.shape[1]} features")
anchor_col = f"PheCode:{MAIN_PHECODE}"
print(f"Anchor ({anchor_col}) non-zero: {(mat_df[anchor_col] > 0).sum()}")
print(
    f"Note count range: {note_df['note_count'].min()} - {note_df['note_count'].max()}"
)

MAP input matrix: 12476 patients x 202 features
Anchor (PheCode:714.1) non-zero: 3477
Note count range: 1 - 103


In [16]:
# ── Run MAP algorithm ─────────────────────────────────────────────────────────
print("Running MAP algorithm (R)...")
map_results = run_map(mat_df, note_df, anchor_col)

n_cases = (map_results["phenotype"] == 1).sum()
n_controls = (map_results["phenotype"] == 0).sum()
print(f"\nMAP results: {n_cases} cases, {n_controls} controls")
print("\nScore distribution:")
print(map_results["score"].describe().to_string())

Running MAP algorithm (R)...

MAP results: 1398 cases, 11078 controls

Score distribution:
count    12476.000000
mean         0.129028
std          0.209651
min          0.000000
25%          0.000000
50%          0.000000
75%          0.411610
max          0.654252


In [17]:
# ── Save cohort with patient demographics ─────────────────────────────────────
map_results.to_parquet(OUTPUT_DIR / "data" / "map_results.parquet", index=False)

cohort = map_results.merge(
    patients, left_on="patient_id", right_on="subject_id", how="left"
)
cohort.to_parquet(OUTPUT_DIR / "data" / "cohort.parquet", index=False)

cases = cohort[cohort["phenotype"] == 1]
controls = cohort[cohort["phenotype"] == 0]

print(f"Cohort saved: {len(cohort)} patients")
print(f"  Cases:    {len(cases)}")
print(f"  Controls: {len(controls)}")

Cohort saved: 12476 patients
  Cases:    1398
  Controls: 11078


---
## Section 4: Characterization

Demographic summaries, CONSORT flow, comorbidity analysis, ONCE feature prevalence
comparison (cases vs controls), and MAP score distribution.

In [18]:
# ── CONSORT Flow ──────────────────────────────────────────────────────────────
total_patients = patients["subject_id"].nunique()
total_in_obs = obs_log["subject_id"].nunique()
candidates_in_cohort = len(cohort)
n_cases = len(cases)
n_controls = len(controls)

consort = {
    "Total patients in MIMIC-IV": total_patients,
    "Patients with structured EHR events": total_in_obs,
    "Candidates with ONCE features (MAP input)": candidates_in_cohort,
    "MAP cases (phenotype=1)": n_cases,
    "MAP controls (phenotype=0)": n_controls,
}
print("=== CONSORT Flow ===")
for step, count in consort.items():
    print(f"  {step}: {count:,}")

with open(OUTPUT_DIR / "data" / "consort_flow.json", "w") as f:
    json.dump(consort, f, indent=2)

=== CONSORT Flow ===
  Total patients in MIMIC-IV: 364,627
  Patients with structured EHR events: 223,233
  Candidates with ONCE features (MAP input): 12,476
  MAP cases (phenotype=1): 1,398
  MAP controls (phenotype=0): 11,078


In [33]:
import plotly.io as pio

pio.renderers.default = "browser"

In [40]:
# ── CONSORT Flow Diagram ───────────────────────────────────────────────────────
n_excl1 = total_patients - total_in_obs  # excluded: no structured EHR
n_excl2 = total_in_obs - candidates_in_cohort  # excluded: no ONCE features

fig_consort = go.Figure()
fig_consort.update_layout(
    title=dict(text="CONSORT Flow — RA Cohort (MAP Phenotyping)", font=dict(size=15)),
    xaxis=dict(
        range=[-0.2, 11.2],
        showgrid=False,
        zeroline=False,
        visible=False,
        fixedrange=True,
    ),
    yaxis=dict(
        range=[1.0, 10.2],
        showgrid=False,
        zeroline=False,
        visible=False,
        fixedrange=True,
    ),
    width=880,
    height=680,
    plot_bgcolor="white",
    margin=dict(l=10, r=10, t=50, b=10),
)


def _box(x0, y0, x1, y1, html, fill="#D6EAF8", size=11):
    fig_consort.add_shape(
        type="rect",
        x0=x0,
        y0=y0,
        x1=x1,
        y1=y1,
        line=dict(color="#2C3E50", width=1.5),
        fillcolor=fill,
    )
    fig_consort.add_annotation(
        x=(x0 + x1) / 2,
        y=(y0 + y1) / 2,
        text=html,
        showarrow=False,
        xref="x",
        yref="y",
        font=dict(size=size, color="#2C3E50"),
        align="center",
    )


def _line(x0, y0, x1, y1):
    fig_consort.add_shape(
        type="line",
        x0=x0,
        y0=y0,
        x1=x1,
        y1=y1,
        line=dict(color="#2C3E50", width=2),
        xref="x",
        yref="y",
    )


def _arrow(x_tail, y_tail, x_head, y_head):
    fig_consort.add_annotation(
        ax=x_tail,
        ay=y_tail,
        x=x_head,
        y=y_head,
        xref="x",
        yref="y",
        axref="x",
        ayref="y",
        showarrow=True,
        arrowhead=2,
        arrowsize=1.2,
        arrowwidth=2,
        arrowcolor="#2C3E50",
        text="",
    )


# ── Main flow boxes (centered at x=4.5, spanning [2.0, 7.0]) ─────────────────
MX = 4.5

_box(2.0, 8.75, 7.0, 9.5, f"<b>All MIMIC-IV Patients</b><br>n = {total_patients:,}")

_box(
    2.0,
    6.45,
    7.0,
    7.2,
    f"<b>Patients with Structured EHR Events</b><br>n = {total_in_obs:,}",
)

_box(
    2.0,
    4.15,
    7.0,
    4.9,
    f"<b>Candidates with ONCE Features</b><br>(MAP Input)  n = {candidates_in_cohort:,}",
)

_box(
    0.3,
    1.7,
    4.0,
    2.7,
    f"<b>RA Cases</b><br>MAP phenotype = 1<br>n = {n_cases:,}",
    fill="#D5F5E3",
)

_box(
    5.0,
    1.7,
    8.7,
    2.7,
    f"<b>Controls</b><br>MAP phenotype = 0<br>n = {n_controls:,}",
    fill="#FDEBD0",
)

# ── Exclusion boxes (right side, vertically centred on flow midpoints) ────────
mid1_y = (8.75 + 7.2) / 2  # 7.975
mid2_y = (6.45 + 4.9) / 2  # 5.675

_box(
    7.5,
    mid1_y - 0.375,
    10.7,
    mid1_y + 0.375,
    f"<b>Excluded</b>  n = {n_excl1:,}<br>No structured EHR events",
    fill="#FADBD8",
    size=10,
)

_box(
    7.5,
    mid2_y - 0.375,
    10.7,
    mid2_y + 0.375,
    f"<b>Excluded</b>  n = {n_excl2:,}<br>No ONCE features",
    fill="#FADBD8",
    size=10,
)

# ── Connectors ────────────────────────────────────────────────────────────────
# Box 1 → midpoint → Box 2  (+  branch right to Excl 1)
_line(MX, 8.75, MX, mid1_y)
_arrow(MX, mid1_y, MX, 7.2)
_arrow(MX, mid1_y, 7.5, mid1_y)

# Box 2 → midpoint → Box 3  (+  branch right to Excl 2)
_line(MX, 6.45, MX, mid2_y)
_arrow(MX, mid2_y, MX, 4.9)
_arrow(MX, mid2_y, 7.5, mid2_y)

# Box 3 → split → Cases & Controls
split_y = 3.2
cases_cx = (0.3 + 4.0) / 2  # 2.15
ctrl_cx = (5.0 + 8.7) / 2  # 6.85

_line(MX, 4.15, MX, split_y)
_line(MX, split_y, cases_cx, split_y)
_arrow(cases_cx, split_y, cases_cx, 2.7)
_line(MX, split_y, ctrl_cx, split_y)
_arrow(ctrl_cx, split_y, ctrl_cx, 2.7)

fig_consort.write_json(str(OUTPUT_DIR / "plots" / "consort_flow_diagram.json"))
fig_consort.show()

In [34]:
# ── Age Distribution ──────────────────────────────────────────────────────────
fig_age = px.histogram(
    cohort,
    x="anchor_age",
    color=cohort["phenotype"].map({1: "Case", 0: "Control"}),
    nbins=30,
    barmode="overlay",
    opacity=0.7,
    title="Age Distribution: RA Cases vs Controls",
    labels={"anchor_age": "Age (anchor_age)", "color": "Group"},
)
fig_age.write_json(str(OUTPUT_DIR / "plots" / "age_distribution.json"))
fig_age.show()

print(
    "Age — Cases:   median={:.0f}, IQR=[{:.0f}, {:.0f}]".format(
        cases["anchor_age"].median(),
        cases["anchor_age"].quantile(0.25),
        cases["anchor_age"].quantile(0.75),
    )
)
print(
    "Age — Controls: median={:.0f}, IQR=[{:.0f}, {:.0f}]".format(
        controls["anchor_age"].median(),
        controls["anchor_age"].quantile(0.25),
        controls["anchor_age"].quantile(0.75),
    )
)

Age — Cases:   median=68, IQR=[57, 78]
Age — Controls: median=67, IQR=[55, 78]


In [35]:
# ── Sex Distribution ──────────────────────────────────────────────────────────
sex_data = (
    cohort.groupby(["gender", cohort["phenotype"].map({1: "Case", 0: "Control"})])
    .size()
    .reset_index(name="count")
)
sex_data.columns = ["Gender", "Group", "Count"]
fig_sex = px.bar(
    sex_data,
    x="Count",
    y="Gender",
    color="Group",
    barmode="group",
    orientation="h",
    title="Sex Distribution: RA Cases vs Controls",
)
fig_sex.write_json(str(OUTPUT_DIR / "plots" / "sex_distribution.json"))
fig_sex.show()

In [36]:
# ── Race Distribution ─────────────────────────────────────────────────────────
first_adm = (
    admissions.sort_values("admittime").groupby("subject_id").first().reset_index()
)
cohort_race = cohort.merge(
    first_adm[["subject_id", "race"]],
    left_on="patient_id",
    right_on="subject_id",
    how="left",
)

race_data = (
    cohort_race.groupby(
        ["race", cohort_race["phenotype"].map({1: "Case", 0: "Control"})]
    )
    .size()
    .reset_index(name="count")
)
race_data.columns = ["Race", "Group", "Count"]
top_races = race_data.groupby("Race")["Count"].sum().nlargest(8).index
race_data = race_data[race_data["Race"].isin(top_races)]
fig_race = px.bar(
    race_data,
    x="Count",
    y="Race",
    color="Group",
    barmode="group",
    orientation="h",
    title="Race Distribution: RA Cases vs Controls",
)
fig_race.write_json(str(OUTPUT_DIR / "plots" / "race_distribution.json"))
fig_race.show()

In [37]:
# ── Top Comorbidities (Cases) ─────────────────────────────────────────────────
case_ids = set(cases["patient_id"])
case_obs = obs_log_with_nlp[
    (obs_log_with_nlp["subject_id"].isin(case_ids))
    & (obs_log_with_nlp["event_type"] == "phecode")
]
case_obs = case_obs[case_obs["event"] != f"PheCode:{MAIN_PHECODE}"]
top_comorbid = (
    case_obs.groupby("event")["subject_id"].nunique().nlargest(15).reset_index()
)
top_comorbid.columns = ["PheCode", "Patients"]
fig_comorbid = px.bar(
    top_comorbid,
    x="Patients",
    y="PheCode",
    orientation="h",
    title="Top 15 Comorbidities in RA Cases (by patient count)",
)
fig_comorbid.update_layout(yaxis=dict(autorange="reversed"))
fig_comorbid.write_json(str(OUTPUT_DIR / "plots" / "top_comorbidities.json"))
fig_comorbid.show()

print("Top 5 comorbidities in cases:")
for _, row in top_comorbid.head(5).iterrows():
    print(f"  {row['PheCode']}: {row['Patients']} patients")

Top 5 comorbidities in cases:
  PheCode:401.1: 951 patients
  PheCode:272.1: 784 patients
  PheCode:530.11: 756 patients
  PheCode:318: 733 patients
  PheCode:585.1: 603 patients


In [38]:
# ── ONCE Feature Prevalence: Cases vs Controls ────────────────────────────────
nlp_events = {f"CUI:{item['cui']}" for item in once_features["nlp_target_cuis"]}
all_once = once_events | nlp_events

once_obs = obs_log_with_nlp[obs_log_with_nlp["event"].isin(all_once)]

case_prev = once_obs[once_obs["subject_id"].isin(case_ids)].groupby("event")[
    "subject_id"
].nunique() / len(cases)
control_ids = set(controls["patient_id"])
ctrl_prev = once_obs[once_obs["subject_id"].isin(control_ids)].groupby("event")[
    "subject_id"
].nunique() / len(controls)

prev_df = pd.DataFrame({"Cases": case_prev, "Controls": ctrl_prev}).fillna(0)
prev_df["Diff"] = prev_df["Cases"] - prev_df["Controls"]
prev_df = prev_df.nlargest(15, "Diff").reset_index()
prev_df.columns = ["Feature", "Cases", "Controls", "Diff"]

fig_prev = go.Figure()
fig_prev.add_trace(
    go.Bar(y=prev_df["Feature"], x=prev_df["Cases"], name="Cases", orientation="h")
)
fig_prev.add_trace(
    go.Bar(
        y=prev_df["Feature"], x=prev_df["Controls"], name="Controls", orientation="h"
    )
)
fig_prev.update_layout(
    title="Top 15 ONCE Features: Cases vs Controls (prevalence)",
    xaxis_title="Prevalence",
    barmode="group",
    yaxis=dict(autorange="reversed"),
)
fig_prev.write_json(str(OUTPUT_DIR / "plots" / "once_feature_prevalence.json"))
fig_prev.show()

In [39]:
# ── MAP Score Distribution ────────────────────────────────────────────────────
fig_score = px.histogram(
    cohort,
    x="score",
    nbins=50,
    title="MAP Score Distribution",
    labels={"score": "MAP Posterior Probability"},
)
fig_score.write_json(str(OUTPUT_DIR / "plots" / "map_score_distribution.json"))
fig_score.show()

print("\nAll plots saved to plots/.")
print("CONSORT flow saved to data/consort_flow.json.")
print("Done.")


All plots saved to plots/.
CONSORT flow saved to data/consort_flow.json.
Done.


In [41]:
# ── Save final RA cohort patient IDs ─────────────────────────────────────────
cohort_ids = cohort[["patient_id", "phenotype"]].copy()
cohort_ids.to_csv(OUTPUT_DIR / "data" / "ra_cohort_patient_ids.csv", index=False)

cases["patient_id"].to_csv(
    OUTPUT_DIR / "data" / "ra_cases_patient_ids.csv", index=False
)
controls["patient_id"].to_csv(
    OUTPUT_DIR / "data" / "ra_controls_patient_ids.csv", index=False
)

print(
    f"Saved ra_cohort_patient_ids.csv  — {len(cohort_ids):,} patients (cases + controls)"
)
print(f"Saved ra_cases_patient_ids.csv   — {len(cases):,} cases")
print(f"Saved ra_controls_patient_ids.csv — {len(controls):,} controls")

Saved ra_cohort_patient_ids.csv  — 12,476 patients (cases + controls)
Saved ra_cases_patient_ids.csv   — 1,398 cases
Saved ra_controls_patient_ids.csv — 11,078 controls


In [43]:
# ── Cohort Overlap: Current (MAP) vs Baseline ─────────────────────────────────
CURRENT_IDS_FILE = PROJECT_ROOT / "notebooks" / "ra_cohort_patient_ids.csv"
BASELINE_IDS_FILE = PROJECT_ROOT / "notebooks" / "ra_cohort_patient_ids_base.csv"

current_ids = set(pd.read_csv(CURRENT_IDS_FILE)["patient_id"])
baseline_ids = set(pd.read_csv(BASELINE_IDS_FILE)["subject_id"])

only_current = current_ids - baseline_ids
only_baseline = baseline_ids - current_ids
shared = current_ids & baseline_ids

print(f"Current (MAP) cohort: {len(current_ids):,}")
print(f"Baseline cohort:      {len(baseline_ids):,}")
print(f"Shared:               {len(shared):,}")
print(f"Only in current:      {len(only_current):,}")
print(f"Only in baseline:     {len(only_baseline):,}")

# ── Venn diagram (plotly) ─────────────────────────────────────────────────────
jaccard = len(shared) / len(current_ids | baseline_ids)
r = 1.8
cx_gap = 1.6 + (1 - jaccard)

cx_L, cx_R, cy = -cx_gap / 2, cx_gap / 2, 0.0

fig_venn = go.Figure()
fig_venn.update_layout(
    title=dict(text="Patient Overlap: MAP Cohort vs Baseline", font=dict(size=15)),
    xaxis=dict(
        range=[-4, 4], showgrid=False, zeroline=False, visible=False, scaleanchor="y"
    ),
    yaxis=dict(range=[-3, 3], showgrid=False, zeroline=False, visible=False),
    width=700,
    height=480,
    plot_bgcolor="white",
    margin=dict(l=10, r=10, t=55, b=10),
)

for cx, fill in [
    (cx_L, "rgba(93,173,226,0.45)"),
    (cx_R, "rgba(235,152,78,0.45)"),
]:
    fig_venn.add_shape(
        type="circle",
        x0=cx - r,
        y0=cy - r,
        x1=cx + r,
        y1=cy + r,
        line=dict(color="#2C3E50", width=2),
        fillcolor=fill,
    )

mid_x = (cx_L + cx_R) / 2
fig_venn.add_annotation(
    x=cx_L - r * 0.45,
    y=0,
    text=f"<b>{len(only_current):,}</b><br>only MAP",
    showarrow=False,
    font=dict(size=13),
    align="center",
)
fig_venn.add_annotation(
    x=mid_x,
    y=0,
    text=f"<b>{len(shared):,}</b><br>shared",
    showarrow=False,
    font=dict(size=13),
    align="center",
)
fig_venn.add_annotation(
    x=cx_R + r * 0.45,
    y=0,
    text=f"<b>{len(only_baseline):,}</b><br>only baseline",
    showarrow=False,
    font=dict(size=13),
    align="center",
)

fig_venn.add_annotation(
    x=cx_L, y=r + 0.3, text="<b>MAP</b>", showarrow=False, font=dict(size=12)
)
fig_venn.add_annotation(
    x=cx_R, y=r + 0.3, text="<b>Baseline</b>", showarrow=False, font=dict(size=12)
)

fig_venn.add_annotation(
    x=0,
    y=-(r + 0.6),
    text=f"Jaccard: {jaccard:.3f}   |   "
    f"Overlap / MAP: {len(shared) / len(current_ids):.1%}   |   "
    f"Overlap / baseline: {len(shared) / len(baseline_ids):.1%}",
    showarrow=False,
    font=dict(size=11, color="#555"),
)

fig_venn.write_json(str(OUTPUT_DIR / "plots" / "cohort_overlap_venn.json"))
fig_venn.show()

Current (MAP) cohort: 1,398
Baseline cohort:      3,521
Shared:               1,395
Only in current:      3
Only in baseline:     2,126
